# Loan Approval Dashboard

using a dataset from Kaggle.  
Columns:  
person_age	   
person_gender  
person_education  
person_income  
person_emp_exp  
person_home_ownership  
loan_amnt  
loan_intent	loan_int_rate  
loan_percent_income  
cb_person_cred_hist_length  
credit_score  
previous_loan_defaults_on_file  
loan_status


In [842]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import functools, operator

In [843]:
data = pd.read_csv('au_loan_data.csv')
data


,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,help_debt_amount,loan_amnt,loan_intent,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,application_date
0,50,Female,Doctorate,WA,53716,Unemployed,4,RENT,0,0,13300,HOME_IMPROVEMENT,60,17.32,0.2476,32,713,No,0,2023-06-15
1,41,Male,Certificate/Diploma,VIC,20000,Full-time,9,RENT,0,0,30200,DEBT_CONSOLIDATION,36,11.59,1.5100,22,845,No,0,2025-12-26
2,40,Male,Postgraduate,SA,34133,Casual,8,MORTGAGE,0,0,22200,PERSONAL,36,17.32,0.6504,20,755,No,1,2024-11-01
3,40,Female,Bachelor,WA,71808,Full-time,2,OWN,1,19626,15800,DEBT_CONSOLIDATION,60,10.51,0.2200,18,752,No,1,2022-08-05
4,71,Female,Bachelor,VIC,36560,Full-time,10,RENT,0,0,10600,DEBT_CONSOLIDATION,36,17.24,0.2899,51,655,No,0,2024-03-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,39,Male,Bachelor,NSW,79355,Part-time,9,MORTGAGE,1,23714,32700,VEHICLE,60,14.75,0.4121,18,814,No,1,2023-02-04
44996,75,Female,Certificate/Diploma,QLD,41076,Full-time,7,OWN,0,0,13300,DEBT_CONSOLIDATION,12,17.93,0.3238,55,688,Yes,0,2023-10-01
44997,32,Male,High School,NSW,40313,Part-time,4,FAMILY,0,0,10200,HOME_IMPROVEMENT,72,14.45,0.2530,12,905,No,1,2024-08-17
44998,49,Male,Certificate/Diploma,NSW,28757,Part-time,7,OWN,0,0,14300,VEHICLE,36,16.68,0.4973,30,658,No,0,2022-12-13


### Dataset overview:

In [844]:
data.shape

(45000, 20)

In [845]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  int64  
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_state                    45000 non-null  str    
 4   person_income                   45000 non-null  int64  
 5   person_emp_status               45000 non-null  str    
 6   person_emp_exp                  45000 non-null  int64  
 7   person_home_ownership           45000 non-null  str    
 8   has_help_debt                   45000 non-null  int64  
 9   help_debt_amount                45000 non-null  int64  
 10  loan_amnt                       45000 non-null  int64  
 11  loan_intent                     45000 non-null  str    
 12  loan_term_months                45000 non-n

In [846]:
data.head()

,person_age,person_gender,person_education,person_state,person_income,person_emp_status,person_emp_exp,person_home_ownership,has_help_debt,help_debt_amount,loan_amnt,loan_intent,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status,application_date
0,50,Female,Doctorate,WA,53716,Unemployed,4,RENT,0,0,13300,HOME_IMPROVEMENT,60,17.32,0.2476,32,713,No,0,2023-06-15
1,41,Male,Certificate/Diploma,VIC,20000,Full-time,9,RENT,0,0,30200,DEBT_CONSOLIDATION,36,11.59,1.5100,22,845,No,0,2025-12-26
2,40,Male,Postgraduate,SA,34133,Casual,8,MORTGAGE,0,0,22200,PERSONAL,36,17.32,0.6504,20,755,No,1,2024-11-01
3,40,Female,Bachelor,WA,71808,Full-time,2,OWN,1,19626,15800,DEBT_CONSOLIDATION,60,10.51,0.2200,18,752,No,1,2022-08-05
4,71,Female,Bachelor,VIC,36560,Full-time,10,RENT,0,0,10600,DEBT_CONSOLIDATION,36,17.24,0.2899,51,655,No,0,2024-03-31


In [847]:
data.describe()

,person_age,person_income,person_emp_exp,has_help_debt,help_debt_amount,loan_amnt,loan_term_months,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000
mean,45.203067,69671.052511,7.902822,0.261289,5624.383333,19898.511111,44.193067,13.868166,0.380636,25.213200,750.014200,0.537956
std,12.964998,41039.779524,5.074021,0.439342,11867.780144,13531.481566,16.951981,3.482621,0.358898,13.041812,159.552923,0.498563
min,19.000000,20000.000000,0.000000,0.000000,0.000000,2000.000000,12.000000,5.500000,0.007500,0.000000,58.000000,0.000000
25%,35.000000,41341.500000,4.000000,0.000000,0.000000,10400.000000,36.000000,11.490000,0.152300,15.000000,642.000000,0.000000
50%,43.000000,60041.500000,7.000000,0.000000,0.000000,16200.000000,36.000000,13.860000,0.271000,23.000000,751.000000,1.000000
75%,53.000000,86726.250000,11.000000,1.000000,6552.000000,25300.000000,60.000000,16.230000,0.479200,33.000000,859.000000,1.000000
max,75.000000,450000.000000,42.000000,1.000000,120000.000000,75000.000000,84.000000,25.000000,3.750000,57.000000,1200.000000,1.000000


In [848]:
#clearing whitespace 
for col in  ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']:
    data[col] = data[col].astype(str).str.strip()

#changing column names to make it easierr to find later
data.rename(columns={'cb_person_cred_hist_length': 'credit_history_length'}, inplace=True)

### Pruning the dataset before filling in nulls:

In [849]:
# taking care of duplicate applications
print("Duplicate rows: ", data.duplicated().sum())
data = data.drop_duplicates()

Duplicate rows:  0


In [850]:
# Datatype validation

# integers first:
int_cols = ['person_age','person_emp_exp','loan_amnt','loan_status','credit_history_length','has_help_debt','help_debt_amount','credit_score','loan_term_months']

for col in int_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# floats
float_cols = ['person_income','loan_int_rate','loan_percent_income']

for col in float_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# dates
date_cols = ['application_date']

for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors='coerce')

# categories
cat_cols = ['person_gender','person_education','person_state','person_emp_status','person_home_ownership','loan_intent','previous_loan_defaults_on_file']

for col in cat_cols:
    data[col] = (data[col].astype(str).str.strip().str.upper())

In [851]:
# null check after conversion
print(data.isnull().sum())

person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
dtype: int64


In [852]:
# drop nulls
data.dropna(inplace=True)

In [853]:
# verify datatypes
print(data.dtypes)

person_age                                 int64
person_gender                                str
person_education                             str
person_state                                 str
person_income                              int64
person_emp_status                            str
person_emp_exp                             int64
person_home_ownership                        str
has_help_debt                              int64
help_debt_amount                           int64
loan_amnt                                  int64
loan_intent                                  str
loan_term_months                           int64
loan_int_rate                            float64
loan_percent_income                      float64
credit_history_length                      int64
credit_score                               int64
previous_loan_defaults_on_file               str
loan_status                                int64
application_date                  datetime64[us]
dtype: object


In [854]:
#Invalid / Zero value loan requests
data = data[data['loan_amnt'] > 0]

In [855]:
#credit score check acc to australian standards
data = data[(data['credit_score'] >0 ) & (data['credit_score'] < 1200)]

In [856]:
#credit history check
data = data[(data['credit_history_length']>=0) & (data['credit_history_length'] < 60) ]

In [857]:
#realistic age check
data = data[(data['person_age']>=18) & (data['person_age']<=100)]

In [858]:
#Invalid / Zero value income
data = data[(data['person_income'] > 0)]

In [859]:
# realistic experience check
data = data[(data['person_emp_exp']>=0) & (data['person_emp_exp']<=70)]

In [860]:
data = data[(data['help_debt_amount']>=0) & (data['help_debt_amount']<=150000)]

In [861]:
# allowed category values

valid_gender = ['MALE','FEMALE','NON-BINARY']

valid_education = ['HIGH SCHOOL','CERTIFICATE/DIPLOMA','BACHELOR','POSTGRADUATE','DOCTORATE']

valid_states = ['NSW','VIC','QLD','WA','SA','TAS','ACT','NT']

valid_emp_status = ['FULL-TIME','PART-TIME','SELF-EMPLOYED','CASUAL','UNEMPLOYED']

valid_home_ownership = ['RENT','MORTGAGE','OWN','FAMILY','OTHER']

valid_loan_intent = ['VEHICLE','DEBT_CONSOLIDATION','HOME_IMPROVEMENT','MEDICAL','HOLIDAY','EDUCATION','PERSONAL']

valid_defaults = ['YES','NO']

In [862]:
# filter out invalid categories
data = data[data['person_gender'].isin(valid_gender)]

In [863]:
data = data[data['person_education'].isin(valid_education)]

In [864]:
data = data[data['person_state'].isin(valid_states)]

In [865]:
data = data[data['person_emp_status'].isin(valid_emp_status)]

In [866]:
data = data[data['person_home_ownership'].isin(valid_home_ownership)]

In [867]:
data = data[data['loan_intent'].isin(valid_loan_intent)]

In [868]:
data = data[data['previous_loan_defaults_on_file'].isin(valid_defaults)]

In [869]:
data.shape

(44888, 20)

### Checking Nulls and filling data

In [870]:
data.isnull().sum()
#no null values because the dataset used is a synthetic one.
#still I've added checks for filling in null values

person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
dtype: int64

In [871]:
#Interest rate check
if data['loan_int_rate'].isnull().sum() > 0:
    median_interest_rate = data['loan_interest_rate'].median()
    data['loan_int_rate'].fillna(median_interest_rate, inplace=True)

### Derived values

In [872]:
#Loan to income ratio recalculation since we dropped some values
data['loan_percent_income'] = data['loan_amnt'] / data['person_income'].round(2)

In [873]:
#Income tiers
data['income_tier'] = pd.cut(data['person_income'], bins = [0, 50000, 100000, 200000, 500000], labels=['LOW', 'MIDDLE', 'UPPER-MIDDLE', 'HIGH'])

In [874]:
# income to loan ratio bands
data['dti_band'] = pd.cut(data['loan_percent_income'], bins = [0, 0.15, 0.35, 1.0, 5.0], labels=['LOW', 'MIDDLE', 'HIGH', 'VERY HIGH'])

In [875]:
# age groups
data['age_group'] = pd.cut(data['person_age'], bins = [18, 25, 35, 50, 100], labels=['YOUNG ADULT', 'ADULT', 'MIDDLE-AGED', 'SENIOR'])

In [876]:
# credit score categories
data['credit_score_category'] = pd.cut(data['credit_score'], bins = [0, 460, 661, 735, 853, 1200], labels=['BELOW AVG', 'AVG', 'GOOD', 'VERY GOOD', 'EXCELLENT'])

In [877]:
#fraud flags
fraud_flag = []

# Income vs loan amount mismatch
fraud_flag.append(data['loan_percent_income'] > 0.5)

# Employment experience impossible given age
fraud_flag.append(data['person_emp_exp'] >= data['person_age'] - 16)

# Credit history longer than working life
fraud_flag.append(data['credit_history_length'] >= data['person_age'] - 16)

# Suspiciously round high income (common fraudulent entries)
fraud_flag.append((data['person_income'] % 10000 == 0) & (data['person_income'] > 100000))

#very high loan request compared to income
fraud_flag.append(data['loan_amnt'] > (data['person_income'] * 0.5))

#previously defaulted but approved now
fraud_flag.append((data['previous_loan_defaults_on_file']=='YES') & (data['loan_status']== 1))

#unemployed yet approved for loan
fraud_flag.append((data['person_emp_status']=='UNEMPLOYED') & (data['loan_status']== 1))

#very high debt-to-income ratio
fraud_flag.append((data['loan_percent_income'] > 0.7))

data['fraud_flag'] = functools.reduce(operator.or_, fraud_flag).astype(int)
data['fraud_flag_count'] = sum(f.astype(int) for f in fraud_flag)

In [878]:
#sort by loan application date
data = data.sort_values('application_date').reset_index(drop=True)

In [879]:
#id
data.insert(0, 'id', range(100000, len(data) + 100000))
data['id'] = data['id'].astype(str)

### Final look over

In [880]:
data.describe()

,person_age,person_income,person_emp_exp,has_help_debt,help_debt_amount,loan_amnt,loan_term_months,loan_int_rate,loan_percent_income,credit_history_length,credit_score,loan_status,application_date,fraud_flag,fraud_flag_count
count,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888.000000,44888,44888.000000,44888.000000
mean,45.207828,69673.065920,7.901332,0.261317,5624.894382,19899.095527,44.190251,13.867710,0.380650,25.217965,748.891441,0.537538,2023-12-27 17:37:09.549100,0.296872,0.668464
min,19.000000,20000.000000,0.000000,0.000000,0.000000,2000.000000,12.000000,5.500000,0.007456,0.000000,58.000000,0.000000,2022-01-01 00:00:00,0.000000,0.000000
25%,35.000000,41345.000000,4.000000,0.000000,0.000000,10400.000000,36.000000,11.490000,0.152275,15.000000,641.750000,0.000000,2022-12-27 00:00:00,0.000000,0.000000
50%,43.000000,60042.500000,7.000000,0.000000,0.000000,16200.000000,36.000000,13.860000,0.271006,23.000000,750.000000,1.000000,2023-12-25 00:00:00,0.000000,0.000000
75%,53.000000,86725.250000,11.000000,1.000000,6552.000000,25300.000000,60.000000,16.230000,0.479068,33.000000,858.000000,1.000000,2024-12-28 00:00:00,1.000000,1.000000
max,75.000000,450000.000000,42.000000,1.000000,120000.000000,75000.000000,84.000000,25.000000,3.750000,57.000000,1198.000000,1.000000,2025-12-31 00:00:00,1.000000,5.000000
std,12.967694,41047.758576,5.072632,0.439357,11863.580765,13530.712047,16.951138,3.482739,0.358906,13.044230,158.158645,0.498594,NaN,0.456885,1.115693


In [881]:
data['loan_status'].value_counts()

loan_status
1    24129
0    20759
Name: count, dtype: int64

In [882]:
data.groupby('person_state')['loan_status'].mean()

person_state
ACT    0.524968
NSW    0.538831
NT     0.531732
QLD    0.536711
SA     0.534678
TAS    0.550214
VIC    0.537872
WA     0.536699
Name: loan_status, dtype: float64

In [883]:
data.groupby('income_tier')['loan_status'].mean()

income_tier
LOW             0.413213
MIDDLE          0.575853
UPPER-MIDDLE    0.694547
HIGH            0.777236
Name: loan_status, dtype: float64

### Final Validation

In [884]:
print(data.shape)

(44888, 27)


In [885]:
print(data.isnull().sum())

id                                0
person_age                        0
person_gender                     0
person_education                  0
person_state                      0
person_income                     0
person_emp_status                 0
person_emp_exp                    0
person_home_ownership             0
has_help_debt                     0
help_debt_amount                  0
loan_amnt                         0
loan_intent                       0
loan_term_months                  0
loan_int_rate                     0
loan_percent_income               0
credit_history_length             0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
application_date                  0
income_tier                       0
dti_band                          0
age_group                         0
credit_score_category             0
fraud_flag                        0
fraud_flag_count                  0
dtype: int64


In [886]:
print(data.duplicated().sum())

0


### Exporting CSVs

In [887]:
loan_details = data[['id', 'application_date', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'loan_term_months', 'loan_status', 'dti_band']]

In [888]:
applicant_details = data[
    [
        'id',
        'person_age',
        'person_gender',
        'person_education',
        'person_income',
        'income_tier',
        'person_emp_status',
        'person_emp_exp',
        'person_home_ownership',
        'person_state'
    ]
]

In [889]:
credit_details = data[
    [
        'id',
        'credit_score',
        'credit_history_length',
        'previous_loan_defaults_on_file',
        'has_help_debt',
        'help_debt_amount'
    ]
]

In [890]:
fact_risk_flags = data[]

SyntaxError: invalid syntax (2765322631.py, line 1)

In [ ]:
loan_details.to_csv('loan_details.csv', index=False)

applicant_details.to_csv('applicant_details.csv', index=False)

credit_details.to_csv('credit_details.csv', index=False)

fact_risk_flags.to_csv('fact_risk_flags.csv', index=False)